# NL2SQL Practice

SCOTT 데이터베이스에 자연어로 질문하고, 생성된 SQL과 실행 결과를 비교한다.

- LLM: 프롬프트에 포함된 스키마와 질문을 바탕으로 SQL 생성
- DB: SQL 실행 및 결과 반환
- Practice I: 기본 질의, Join, Subquery, 중복 행
- Practice II: GROUP BY, NULL, ROLLUP/CUBE, 분석 함수


## 0. 준비


### DuckDB

<a href="https://duckdb.org/"><img src="https://raw.githubusercontent.com/duckdb/duckdb/main/logo/DuckDB_Logo-horizontal.png" alt="DuckDB 로고" width="220"></a>

- 분석 작업에 특화된 관계형 데이터베이스(RDBMS)
- 별도 DB 서버 없이 Python 프로그램 안에서 동작하는 임베디드 DB
- 이 실습에서는 데이터를 메모리에 저장하며, DB 연결을 종료하면 데이터가 사라짐
- 파일에 저장하는 방식도 지원하지만, 이 실습에서는 사용하지 않음


In [ ]:
!pip -q install duckdb openai

In [ ]:
import os, re
from getpass import getpass

import duckdb
import pandas as pd
from IPython.display import display
from openai import OpenAI

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

client = OpenAI()
MODEL = "gpt-4o-mini"   # SQL 생성 모델
print("Ready")

### SCOTT 데이터베이스

DuckDB의 메모리 DB에 SCOTT 데이터를 저장한다.

- **EMP**: 직원 정보
- **DEPT**: 부서 정보


In [ ]:
con = duckdb.connect()   # in-memory database

con.execute("""
CREATE OR REPLACE TABLE DEPT (
  DEPTNO INTEGER PRIMARY KEY,
  DNAME  VARCHAR,
  LOC    VARCHAR
);
CREATE OR REPLACE TABLE EMP (
  EMPNO    INTEGER PRIMARY KEY,
  ENAME    VARCHAR,
  JOB      VARCHAR,
  MGR      INTEGER,
  HIREDATE DATE,
  SAL      INTEGER,
  COMM     INTEGER,
  DEPTNO   INTEGER
);

INSERT INTO DEPT VALUES
(10,'ACCOUNTING','NEW YORK'),
(20,'RESEARCH','DALLAS'),
(30,'SALES','CHICAGO'),
(40,'OPERATIONS','BOSTON');

INSERT INTO EMP VALUES
(7369,'SMITH','CLERK',7902,'1980-12-17',800,NULL,20),
(7499,'ALLEN','SALESMAN',7698,'1981-02-20',1600,300,30),
(7521,'WARD','SALESMAN',7698,'1981-02-22',1250,500,30),
(7566,'JONES','MANAGER',7839,'1981-04-02',2975,NULL,20),
(7654,'MARTIN','SALESMAN',7698,'1981-09-28',1250,1400,30),
(7698,'BLAKE','MANAGER',7839,'1981-05-01',2850,NULL,30),
(7782,'CLARK','MANAGER',7839,'1981-06-09',2450,NULL,10),
(7788,'SCOTT','ANALYST',7566,'1987-04-19',3000,NULL,20),
(7839,'KING','PRESIDENT',NULL,'1981-11-17',5000,NULL,10),
(7844,'TURNER','SALESMAN',7698,'1981-09-08',1500,0,30),
(7876,'ADAMS','CLERK',7788,'1987-05-23',1100,NULL,20),
(7900,'JAMES','CLERK',7698,'1981-12-03',950,NULL,30),
(7902,'FORD','ANALYST',7566,'1981-12-03',3000,NULL,20),
(7934,'MILLER','CLERK',7782,'1982-01-23',1300,NULL,10);

""")


def run(sql):
    """SQL을 실행하고 결과를 DataFrame으로 반환"""
    return con.sql(sql).df()


run("SELECT * FROM emp")

### NL2SQL 함수

- `build_prompt`: 지시문, 스키마, 힌트, 질문으로 프롬프트 구성
- `call_llm`: LLM에 프롬프트를 전달해 SQL 생성
- `clean_sql`: 생성된 SQL의 문장 수와 금지 키워드 확인
- `ask`: SQL 생성, 실행, 결과 비교


In [ ]:
# LLM에 전달할 Schema 설명
SCHEMA = """
DEPT(deptno, dname, loc)
  -- 부서: 부서번호, 부서명, 위치
EMP(empno, ename, job, mgr, hiredate, sal, comm, deptno)
  -- 직원: 사번, 이름, 직무, 상사의 사번, 입사일(DATE), 급여, 커미션, 부서번호
  -- EMP.deptno 는 DEPT.deptno 를 참조한다.
  -- EMP.mgr 는 상사의 EMP.empno 이다.
""".strip()


def build_prompt(question, schema=SCHEMA, hint=""):
    parts = [
        "질문에 답하는 DuckDB SQL을 작성한다.",
        "규칙:",
        "- SELECT 문(또는 WITH 로 시작하는 SELECT 문) 하나만 작성한다.",
        "- 첫 줄에는 '-- 설명: ' 으로 시작하는 한 줄짜리 한국어 설명을 작성한다.",
        "- SQL 이외의 글이나 마크다운은 작성하지 않는다.",
        "- 출력 컬럼은 질문에서 요청한 순서대로 작성한다.",
        "",
        "[Schema]",
        schema if schema else "(제공되지 않음)",
    ]
    if hint:
        parts += ["", "[Hint]", hint]
    parts += ["", "[질문]", question]
    return "\n".join(parts)


def call_llm(prompt):
    resp = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.choices[0].message.content


BANNED = ["insert", "update", "delete", "drop", "alter", "create", "attach", "copy"]


def clean_sql(text):
    m = re.search(r"```(?:sql)?\s*(.*?)```", text, flags=re.S | re.I)
    sql = (m.group(1) if m else text).strip().rstrip(";").strip()
    body = "\n".join(
        line for line in sql.splitlines() if not line.strip().startswith("--")
    ).strip().lower()
    if ";" in body:
        raise ValueError("SQL은 한 문장만 실행한다.")
    if not body.startswith(("select", "with")):
        raise ValueError("SELECT 문만 실행한다.")
    for kw in BANNED:
        if re.search(rf"\b{kw}\b", body):
            raise ValueError(f"금지 키워드가 포함되어 실행을 중단한다: {kw}")
    return sql


from collections import Counter
from decimal import Decimal
from numbers import Number


def _cell(value):
    if pd.isna(value):
        return ("null",)
    if isinstance(value, bool):
        return ("bool", value)
    if isinstance(value, Number):
        return ("number", Decimal(str(value)))
    if isinstance(value, str):
        return ("text", value)
    return (type(value).__name__, str(value))


def compare(df, gold_df, check_order=False):
    """컬럼 위치를 맞춰 비교한다. 별칭은 무시하고 중복 행은 유지한다."""
    if df.shape != gold_df.shape:
        print("[기준 SQL과 결과 비교] 불일치")
        print(f"  생성 결과: {df.shape[0]}행 × {df.shape[1]}열 / 기준 결과: {gold_df.shape[0]}행 × {gold_df.shape[1]}열")
        return False
    rows = lambda frame: [tuple(_cell(v) for v in row) for row in frame.itertuples(index=False, name=None)]
    actual, expected = rows(df), rows(gold_df)
    same_values = Counter(actual) == Counter(expected)
    same_order = actual == expected
    matched = same_values and (same_order or not check_order)
    print("[기준 SQL과 결과 비교]", "일치" if matched else "불일치")
    if not same_values:
        print("  컬럼 순서와 각 컬럼의 값, 중복 행을 확인한다.")
    elif check_order and not same_order:
        print("  값은 같지만 행 순서가 다르다.")
    return matched


def ask(question, schema=SCHEMA, hint="", show_prompt=False, retry=True, gold=None, check_order=False):
    """자연어 질문을 SQL로 변환해 실행

    question    : 질문
    schema      : LLM에 전달할 Schema 설명 ("" 이면 Schema 없이 질문)
    hint        : 추가로 전달할 규칙이나 용어 설명
    show_prompt : True 이면 LLM에 전달한 Prompt를 출력
    retry       : 실행 오류가 나면 오류 메시지를 LLM에 전달해 한 번 더 생성
    gold        : 기준 SQL (주면 두 결과를 비교)
    check_order : True이면 출력 행 순서도 비교
    """
    prompt = build_prompt(question, schema, hint)
    if show_prompt:
        print("===== Prompt =====")
        print(prompt)
        print("=" * 18, "\n")
    print("질문:", question)

    try:
        sql = clean_sql(call_llm(prompt))
    except ValueError as e:
        print("\n[실행하지 않음]", e)
        return None
    print("\n[생성된 SQL]")
    print(sql)

    try:
        df = run(sql)
    except Exception as e:
        print("\n[실행 오류]", str(e).strip().splitlines()[0])
        if not retry:
            return None
        print("\n오류 메시지를 전달해 SQL을 다시 생성한다.")
        fix_prompt = (
            prompt
            + f"\n\n[이전에 작성한 SQL]\n{sql}"
            + f"\n\n[실행 오류]\n{e}"
            + "\n\n같은 규칙에 따라 오류를 수정한 SQL을 작성한다."
        )
        try:
            sql = clean_sql(call_llm(fix_prompt))
            print("\n[수정된 SQL]")
            print(sql)
            df = run(sql)
        except Exception as e2:
            print("\n[재실행 실패]", str(e2).strip().splitlines()[0])
            return None

    print(f"\n[실행 결과] {len(df)}행")
    if gold is not None:
        compare(df, run(gold), check_order=check_order)
    return df


print("Ready")

---
# Practice I

## 1. 기본 질의

SQL로 작성한 질의와 같은 내용의 자연어 질문을 비교한다.


In [ ]:
# 급여가 2000 이상인 직원 (급여가 높은 순)
run("""
SELECT ename, job, sal
FROM emp
WHERE sal >= 2000
ORDER BY sal DESC
""")

`ask()`는 질문을 SQL로 변환해 실행한다. `show_prompt=True`는 LLM에 전달한 프롬프트를 출력한다.


In [ ]:
ask("급여가 2000 이상인 직원의 이름, 직무, 급여를 급여가 높은 순으로 보여줘", show_prompt=True)

- 입력: 지시문, 스키마, 질문
- LLM 출력: SQL과 첫 줄의 `-- 설명:`
- DB 출력: SQL 실행 결과

이 실습의 프롬프트에는 직원 14명의 실제 데이터가 포함되지 않는다. LLM은 스키마 설명으로 테이블과 컬럼의 의미를 파악한다.

### 스키마가 없는 경우

`schema=""`로 같은 질문을 전달한다.


In [ ]:
ask("급여가 2000 이상인 직원의 이름과 급여를 보여줘", schema="", retry=False)

스키마가 없으면 LLM이 테이블과 컬럼 이름을 추측할 수 있다. 실제 DB에 없는 이름을 사용하면 실행 오류가 발생한다.

**Schema Linking**: 질문의 표현을 실제 테이블과 컬럼에 연결하는 과정. 예를 들어 ‘급여’를 `EMP.sal`에 연결한다.


## 2. Join

`gold=`에 기준 SQL을 지정해 생성된 SQL의 결과와 비교한다. 컬럼은 요청한 순서대로 비교하며, 별칭은 무시한다. 중복 행은 구분하고, 정렬이 필요한 문제는 `check_order=True`로 행 순서도 비교한다.

- ‘일치’는 현재 데이터의 결과가 같다는 뜻이며, 특정 SQL 문법의 사용 여부까지 판단하지는 않는다.

- **Equi-Join**: 두 컬럼의 값이 같은 행 연결
- **Self-Join**: 같은 테이블을 서로 다른 역할로 사용
- **조인과 필터**: 테이블을 연결한 뒤 조건에 맞는 행 선택

### 2-1. Equi-Join


In [ ]:
GOLD_EQUI = """
SELECT ename, loc
FROM emp, dept
WHERE emp.deptno = dept.deptno
"""

ask("각 직원의 이름과 근무하는 부서의 위치를 보여줘", gold=GOLD_EQUI)

부서 위치(`loc`)는 DEPT에 있다. `emp.deptno = dept.deptno`로 직원과 소속 부서를 연결한다.

### 2-2. Self-Join


In [ ]:
GOLD_MGR = """
SELECT e1.ename AS emp, e2.ename AS mgr
FROM emp e1, emp e2
WHERE e1.mgr = e2.empno
"""

ask("상사가 있는 직원의 이름과 직속 상사의 이름을 보여줘", gold=GOLD_MGR)

EMP를 직원(`e1`)과 상사(`e2`) 역할로 각각 사용한다.
상사가 없는 KING(`mgr IS NULL`)을 제외한 13행이 반환된다.

### 2-3. 조인과 필터


In [ ]:
GOLD_JOIN_FILTER = """
SELECT e.ename, d.dname
FROM emp e, dept d
WHERE e.deptno = d.deptno
  AND d.loc = 'DALLAS'
"""

ask("DALLAS에서 근무하는 직원의 이름과 부서 이름을 보여줘", gold=GOLD_JOIN_FILTER)


부서번호로 EMP와 DEPT를 연결하고, 부서 위치가 DALLAS인 직원만 선택한다.


## 3. Subquery

### Subquery 1)


In [ ]:
GOLD_7566 = """
SELECT ename
FROM emp
WHERE sal > (SELECT sal
             FROM emp
             WHERE empno = 7566)
"""

ask("사번이 7566인 직원보다 급여가 높은 직원의 이름을 보여줘", gold=GOLD_7566)

안쪽 쿼리에서 7566번 직원(JONES)의 급여를 구한 뒤 비교한다.

### Subquery 2)


In [ ]:
GOLD_ANY = """
SELECT empno, ename, job
FROM emp
WHERE sal < ANY (SELECT sal
                 FROM emp
                 WHERE job = 'CLERK')
  AND job <> 'CLERK'
"""

ask(
    "직무가 CLERK이 아닌 직원 중에서, CLERK 중 적어도 한 명보다 급여가 낮은 직원의 사번, 이름, 직무를 보여줘",
    gold=GOLD_ANY,
)

안쪽 쿼리는 CLERK 4명의 급여를 반환한다.

- `< ANY`: 적어도 한 명의 CLERK보다 급여가 낮은지 비교
- `< (SELECT MAX(sal) ...)`: CLERK의 최고 급여보다 낮은지 비교

이 질문에서는 두 방식의 결과가 같다.

- CLERK의 최고 급여는 1300이며, 기준 결과는 WARD·MARTIN 2명
- `< MIN(sal)`은 가장 낮은 급여보다도 낮아야 하므로 이 질문의 조건과 다름


In [ ]:
print("[기준 결과]")
display(run(GOLD_ANY))


In [ ]:
ask(
    "직무가 CLERK이 아닌 직원 중에서, CLERK 중 적어도 한 명보다 급여가 낮은 직원의 사번, 이름, 직무를 보여줘",
    hint="적어도 한 명보다 낮다는 것은 CLERK의 최고 급여보다 낮다는 의미다. ANY를 사용해 작성한다.",
    gold=GOLD_ANY,
)


### Subquery 3)


In [ ]:
GOLD_DNAME = """
SELECT dname
FROM dept d
WHERE EXISTS (SELECT *
              FROM emp e
              WHERE d.deptno = e.deptno)
"""

ask("직원이 한 명 이상 있는 부서의 이름을 보여줘", gold=GOLD_DNAME)

기준 SQL에서 서브쿼리의 `d.deptno = e.deptno`는 바깥 쿼리의 부서(`d`)를 참조한다. 각 부서에 직원이 존재하는지 확인한다.

생성 SQL이 JOIN과 GROUP BY를 사용해도 현재 데이터에서 같은 부서 목록을 구할 수 있다.


## 4. 실행 가능한 SQL과 올바른 SQL

질문: *직원이 한 명 이상 있는 부서의 부서번호*

두 SQL의 실행 결과와 행 수를 비교한다.


In [ ]:
# SQL A: Join
run("""
SELECT d.deptno
FROM dept d, emp e
WHERE d.deptno = e.deptno
""")

In [ ]:
# SQL B: Join + DISTINCT
GOLD_DEPTNO = """
SELECT DISTINCT d.deptno
FROM dept d, emp e
WHERE d.deptno = e.deptno
"""
run(GOLD_DEPTNO)

- SQL A: 직원마다 한 행이 생성되어 부서번호가 중복됨(14행)
- SQL B: 중복을 제거한 부서 목록(3행)

`DISTINCT`, `EXISTS`, `IN` 등으로 중복 없는 부서 목록을 구할 수 있다.


In [ ]:
ask("직원이 한 명 이상 있는 부서의 부서번호를 보여줘", gold=GOLD_DEPTNO)

- 3행 반환: 중복을 제거한 방법 확인
- 14행 반환: 질문에 ‘중복 없이’를 추가해 결과 비교

SQL이 실행된다는 사실만으로 질문에 맞는 결과인지 판단할 수는 없다.


### Practice I 정리

- 스키마와 질문을 바탕으로 LLM이 SQL을 생성하고 DB가 실행한다.
- 테이블·컬럼 연결과 중복 행 처리 여부를 확인한다.
- 생성된 SQL과 결과가 질문의 의도에 맞는지 검증한다.


---
# Practice II

GROUP BY, NULL, ROLLUP/CUBE, 분석 함수를 사용하는 질의를 다룬다.


In [ ]:
ask("부서 이름별 평균 급여를 구해서 평균이 높은 부서부터 보여줘", show_prompt=True)

## 5. GROUP BY와 NULL

- `GROUP BY`와 `HAVING`: 그룹별 집계와 그룹 조건
- NULL: 산술 연산에서의 처리

### 5-1. GROUP BY와 HAVING


In [ ]:
GOLD_HAVING = """
SELECT deptno, MAX(sal) AS max_sal
FROM emp
GROUP BY deptno
HAVING COUNT(*) > 3
"""

ask("직원이 3명보다 많은 부서의 부서번호와 최고 급여를 보여줘", gold=GOLD_HAVING)

‘직원이 3명보다 많은 부서’는 그룹에 대한 조건이다. `GROUP BY deptno`로 묶고 `HAVING COUNT(*) > 3`으로 조건을 지정한다.


### 5-2. NULL이 포함된 계산

질문: *커미션을 포함한 총 급여가 가장 높은 5명*

`sal + comm`으로 계산한 결과를 확인한다.


In [ ]:
run("""
SELECT ename, sal + comm AS total_pay
FROM emp
ORDER BY total_pay DESC
LIMIT 5
""")

`comm`이 NULL이면 `sal + comm`도 NULL이다. KING의 총 급여도 NULL로 계산된다.

- 현재 데이터에서 총 급여가 숫자인 직원은 4명
- 다섯 번째 행은 NULL인 직원 중 하나이며, 동점 정렬 기준이 없어 직원이 고정되지 않음

커미션의 NULL을 0으로 처리한 `GOLD_PAY`와 생성 결과를 비교한다.


In [ ]:
GOLD_PAY = """
SELECT ename, sal + COALESCE(comm, 0) AS total_pay
FROM emp
ORDER BY total_pay DESC
LIMIT 5
"""

ask("커미션을 포함한 총 급여가 가장 높은 5명의 이름과 총 급여를 보여줘", gold=GOLD_PAY)

- 결과 일치: `COALESCE` 등 NULL 처리 방식 확인
- 결과 불일치: NULL을 0으로 처리한다는 힌트를 추가해 비교


In [ ]:
ask(
    "커미션을 포함한 총 급여가 가장 높은 5명의 이름과 총 급여를 보여줘",
    hint="comm 이 NULL 이면 0으로 계산한다.",
    gold=GOLD_PAY,
)

## 6. ROLLUP과 CUBE

- **ROLLUP**: 지정한 컬럼 순서에 따라 단계별 소계와 전체 합계 계산
- **CUBE**: 지정한 컬럼의 모든 조합에 대해 집계

### 6-1. ROLLUP


In [ ]:
GOLD_ROLLUP = """
SELECT job, deptno, SUM(sal) AS sum_sal
FROM emp
GROUP BY ROLLUP(job, deptno)
"""

ask(
    "직무(job)와 부서번호(deptno)별 급여 합계를 구하고, 직무별 소계와 전체 합계도 함께 보여줘. "
    "컬럼은 job, deptno, 급여 합계",
    gold=GOLD_ROLLUP,
)

In [ ]:
print("[기준 SQL]")
print(GOLD_ROLLUP)
display(run(GOLD_ROLLUP))


- `deptno`만 NULL: 직무별 소계
- `job`, `deptno` 모두 NULL: 전체 합계(29025)

위 기준 결과의 NULL은 소계와 합계를 표시한다. 여러 `GROUP BY` 결과를 `UNION ALL`로 합쳐도 같은 결과를 구할 수 있다.




### 출력 형태를 명확히 한 질문

- 소계·합계를 추가 컬럼으로 붙였는지, 별도 행으로 표시했는지 비교
- 전체 합계 29025가 포함되는지 확인


In [ ]:
ask(
    "직무(job)와 부서번호(deptno)별 급여 합계에 직무별 소계와 전체 합계를 별도 행으로 추가해 줘. "
    "집계에서 제외된 컬럼은 NULL로 표시하고, 출력은 job, deptno, 급여 합계의 3개 컬럼만 보여줘",
    hint="ROLLUP(job, deptno)를 사용한다.",
    gold=GOLD_ROLLUP,
)


### 6-2. CUBE


In [ ]:
GOLD_CUBE = """
SELECT job, deptno, SUM(sal) AS sum_sal
FROM emp
GROUP BY CUBE(job, deptno)
"""

ask(
    "직무(job)와 부서번호(deptno)별 급여 합계를 구하고, 직무별 소계, 부서별 소계, 전체 합계도 함께 보여줘. "
    "컬럼은 job, deptno, 급여 합계",
    gold=GOLD_CUBE,
)

In [ ]:
print("[기준 SQL]")
print(GOLD_CUBE)
display(run(GOLD_CUBE))


기준 결과는 ROLLUP 15행, CUBE 18행이다. CUBE에는 부서별 소계 3행이 추가된다.


### 출력 형태를 명확히 한 질문

- 소계·합계를 추가 컬럼으로 붙였는지, 별도 행으로 표시했는지 비교
- 전체 합계 29025가 포함되는지 확인


In [ ]:
ask(
    "직무(job)와 부서번호(deptno)별 급여 합계에 직무별 소계, 부서별 소계, 전체 합계를 별도 행으로 추가해 줘. "
    "집계에서 제외된 컬럼은 NULL로 표시하고, 출력은 job, deptno, 급여 합계의 3개 컬럼만 보여줘",
    hint="CUBE(job, deptno)를 사용한다.",
    gold=GOLD_CUBE,
)


## 7. 분석 함수

- **그룹 집계와 윈도 집계**: `GROUP BY`, `OVER (PARTITION BY ...)`
- **순위**: `RANK`, `DENSE_RANK`
- **그룹별 상위 행**: 순위 계산 후 조건에 맞는 행 선택
- **누적 합**: 정렬 기준과 윈도 범위 지정

### 7-1. 그룹 집계와 윈도 집계

두 질의의 결과 행 수를 비교한다.


In [ ]:
GOLD_GROUP = """
SELECT deptno, COUNT(*) AS cnt, SUM(sal) AS sum_sal
FROM emp
GROUP BY deptno
"""

ask("부서번호별 직원 수와 급여 합계를 보여줘", gold=GOLD_GROUP)

In [ ]:
GOLD_OVER = """
SELECT ename, deptno, sal,
       COUNT(*) OVER (PARTITION BY deptno) AS dept_cnt,
       SUM(sal) OVER (PARTITION BY deptno) AS dept_sum_sal
FROM emp
"""

ask(
    "모든 직원의 이름, 부서번호, 급여를 보여주고, "
    "각 행에 그 직원이 속한 부서의 직원 수와 급여 합계를 함께 보여줘",
    gold=GOLD_OVER,
)

In [ ]:
print("[분석 함수를 사용한 기준 SQL]")
print(GOLD_OVER)
display(run(GOLD_OVER))


- `GROUP BY`: 부서별 한 행으로 집계(3행)
- `OVER (PARTITION BY deptno)`: 직원별 행을 유지하며 부서 집계값 추가(14행)



기준 SQL은 윈도 함수를 사용한다. 생성 SQL이 집계 서브쿼리와 JOIN을 사용해도 결과는 같을 수 있다.


In [ ]:
ask(
    "모든 직원의 이름, 부서번호, 급여, 소속 부서의 직원 수, 소속 부서의 급여 합계를 순서대로 보여줘",
    hint="COUNT와 SUM에 OVER (PARTITION BY deptno)를 사용한다.",
    gold=GOLD_OVER,
)


### 7-2. 순위


In [ ]:
GOLD_RANK = """
SELECT ename, deptno, sal,
       RANK()       OVER (PARTITION BY deptno ORDER BY sal DESC) AS rk,
       DENSE_RANK() OVER (PARTITION BY deptno ORDER BY sal DESC) AS dr
FROM emp
"""

ask(
    "각 직원의 이름, 부서번호, 급여와 부서 내 급여 순위(급여가 높은 순)를 보여줘. "
    "순위는 RANK와 DENSE_RANK 두 가지로 모두 보여줘",
    gold=GOLD_RANK,
)

20번 부서에서 SCOTT과 FORD는 급여가 3000으로 같아 공동 1위다.

- `RANK`: 다음 순위는 3위
- `DENSE_RANK`: 다음 순위는 2위

질문에 동점 처리 기준을 명시한다.

### 7-3. 그룹별 상위 행


In [ ]:
GOLD_TOP = """
SELECT deptno, ename, sal
FROM (
    SELECT deptno, ename, sal,
           RANK() OVER (PARTITION BY deptno ORDER BY sal DESC) AS rk
    FROM emp
)
WHERE rk = 1
"""

ask(
    "부서별로 급여가 가장 높은 직원의 부서번호, 이름, 급여를 보여줘. 급여가 같으면 모두 보여줘",
    gold=GOLD_TOP,
)

In [ ]:
print("[분석 함수를 사용한 기준 SQL]")
print(GOLD_TOP)
display(run(GOLD_TOP))


기준 SQL은 `RANK`로 순위를 계산한 뒤 `rk = 1`인 행을 선택한다. 윈도 함수는 같은 단계의 `WHERE` 절에 직접 사용할 수 없다.

생성 SQL이 MAX와 JOIN을 사용해도 동점자를 모두 포함한 4행을 구할 수 있다.


In [ ]:
ask(
    "부서별로 급여가 가장 높은 직원의 부서번호, 이름, 급여를 보여줘. 급여가 같으면 모두 보여줘",
    hint="RANK() OVER (PARTITION BY deptno ORDER BY sal DESC)로 순위를 구한 뒤 1위인 행을 선택한다.",
    gold=GOLD_TOP,
)


### 7-4. 누적 합

질문: *입사일 순서로 급여의 누적 합*

입사일이 같은 JAMES와 FORD(1981-12-03)의 `running_sal`을 비교한다.


In [ ]:
# SQL A: ORDER BY 만 지정 (Default Window = RANGE)
run("""
SELECT ename, hiredate, sal,
       SUM(sal) OVER (ORDER BY hiredate) AS running_sal
FROM emp
ORDER BY hiredate, empno
""")

In [ ]:
# SQL B: ROWS 지정
GOLD_RUNNING = """
SELECT ename, hiredate, sal,
       SUM(sal) OVER (
           ORDER BY hiredate, empno
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
       ) AS running_sal
FROM emp
ORDER BY hiredate, empno
"""
run(GOLD_RUNNING)

이 예제에서 `ORDER BY`만 지정하면 기본 윈도 범위는 `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`다.

- SQL A: 입사일이 같은 행을 함께 합산해 JAMES와 FORD 모두 23625
- SQL B: 입사일과 사번으로 순서를 정하고 한 행씩 누적해 JAMES 20625, FORD 23625


In [ ]:
ask(
    "입사일이 빠른 순서로 직원의 이름, 입사일, 급여, 급여의 누적 합을 보여줘",
    gold=GOLD_RUNNING,
    check_order=True,
)

생성된 SQL의 정렬 기준과 윈도 범위를 확인한다. 한 행씩 누적하려면 입사일이 같은 경우의 순서도 정해야 한다.

첫 질문은 같은 입사일의 처리 방식이 정해져 있지 않다. 기준 결과와 다르다는 이유만으로 생성 SQL을 오답으로 단정하지 않는다.


In [ ]:
ask(
    "입사일이 빠른 순서로(입사일이 같으면 사번 순서로) 직원의 이름, 입사일, 급여, 급여의 누적 합을 보여줘. "
    "누적 합은 한 행씩 더해서, 입사일이 같은 직원도 값이 서로 다르게 해 줘",
    gold=GOLD_RUNNING,
    check_order=True,
)

## 8. 정리

- 스키마와 질문을 바탕으로 LLM이 SQL을 생성하고 DB가 실행한다.
- 중복 행, NULL, 동점, 모호한 조건에 따라 결과가 달라질 수 있다.
- 생성된 SQL과 결과를 함께 검증한다.

| 비교 | 구분 |
|---|---|
| `WHERE` / `HAVING` | 행 조건 / 그룹 조건 |
| `GROUP BY` / `OVER` | 그룹별 행으로 집계 / 개별 행 유지 |
| `RANK` / `DENSE_RANK` | 동점 다음 순위를 건너뜀 / 건너뛰지 않음 |
| `RANGE` / `ROWS` | 정렬 값 기준 범위 / 행 위치 기준 범위 |


---
## 추가 실습

### A. 직원·상사·부서 연결


In [ ]:
GOLD_3JOIN = """
SELECT e.ename, m.ename AS manager, d.dname
FROM emp e, emp m, dept d
WHERE e.mgr = m.empno
  AND e.deptno = d.deptno
  AND d.loc = 'DALLAS'
"""

ask("DALLAS에서 근무하고 상사가 있는 직원의 이름, 상사 이름, 소속 부서 이름을 보여줘", gold=GOLD_3JOIN)


### B. 모호한 질문

‘급여가 높다’의 기준을 LLM이 어떻게 해석했는지 SQL과 설명에서 확인한다.


In [ ]:
ask("급여가 높은 직원을 보여줘")

### C. NOT IN과 NOT EXISTS

질문: *부하 직원이 없는 직원*

두 SQL의 결과를 비교한다.


In [ ]:
# SQL A: NOT IN
run("""
SELECT ename
FROM emp
WHERE empno NOT IN (SELECT mgr FROM emp)
""")

In [ ]:
# SQL B: NOT EXISTS
GOLD_NO_SUB = """
SELECT e.ename
FROM emp e
WHERE NOT EXISTS (SELECT 1 FROM emp s WHERE s.mgr = e.empno)
"""
run(GOLD_NO_SUB)

SQL A는 0행을 반환한다. 서브쿼리에 KING의 `mgr` 값인 NULL이 포함되어 `NOT IN` 조건이 참이 되는 행이 없기 때문이다.


In [ ]:
ask("부하 직원이 없는 직원의 이름을 보여줘", gold=GOLD_NO_SUB)

- 결과 일치: `NOT EXISTS`, `LEFT JOIN ... IS NULL`, NULL을 제외한 `NOT IN` 등 사용 방식 확인
- 0행 반환: `mgr`의 NULL 처리 여부 확인 후 힌트를 추가해 비교


### D. 분석 함수 응용

- `LAG`: 이전 행의 값
- `ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING`: 앞뒤 한 행을 포함한 윈도
- `FIRST_VALUE`: 윈도 범위의 첫 번째 값
- `OVER ()`: 전체 행을 기준으로 집계


In [ ]:
# 1. LAG
GOLD_LAG = """
SELECT empno, ename, sal,
       sal - LAG(sal) OVER (ORDER BY empno) AS diff
FROM emp
ORDER BY empno
"""

ask(
    "사번 순서로 정렬했을 때 각 직원의 급여와 바로 앞 직원의 급여 차이를 보여줘. "
    "컬럼은 사번, 이름, 급여, 급여 차이. 첫 번째 직원의 차이는 NULL",
    gold=GOLD_LAG,
    check_order=True,
)

In [ ]:
# 2. Moving Window
GOLD_SLIDE = """
SELECT empno, sal,
       ROUND(AVG(sal) OVER (ORDER BY empno
                            ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING), 1) AS moving_avg
FROM emp
ORDER BY empno
"""

ask(
    "사번 순서로 정렬했을 때 각 직원에 대해 앞 직원, 본인, 뒤 직원 세 명의 급여 평균을 "
    "소수 첫째 자리까지 보여줘. 첫 직원과 마지막 직원은 존재하는 직원만으로 평균을 계산해. "
    "컬럼은 사번, 급여, 평균이며 결과도 사번 순서로 정렬해",
    gold=GOLD_SLIDE,
    check_order=True,
)

In [ ]:
print("[분석 함수를 사용한 기준 SQL]")
print(GOLD_SLIDE)
display(run(GOLD_SLIDE))


- LAG·LEAD로 앞뒤 값을 가져와 계산해도 현재 데이터에서는 같은 평균을 구할 수 있다.
- 기준 SQL은 AVG와 ROWS로 계산 범위를 지정한다.


In [ ]:
ask(
    "사번 순서로 앞 직원, 본인, 뒤 직원의 급여 평균을 소수 첫째 자리까지 보여줘. "
    "양끝에서는 존재하는 직원만 포함해. 컬럼은 사번, 급여, 평균이며 결과도 사번 순서로 정렬해",
    hint="AVG(sal) OVER (ORDER BY empno ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING)를 사용한다.",
    gold=GOLD_SLIDE,
    check_order=True,
)


In [ ]:
# 3. FIRST_VALUE
GOLD_FIRST = """
SELECT ename, deptno, sal,
       FIRST_VALUE(sal) OVER (PARTITION BY deptno ORDER BY sal DESC) - sal AS gap
FROM emp
"""

ask(
    "각 직원의 이름, 부서번호, 급여와, 부서 내 최고 급여와 본인 급여의 차이를 보여줘",
    gold=GOLD_FIRST,
)

In [ ]:
print("[분석 함수를 사용한 기준 SQL]")
print(GOLD_FIRST)
display(run(GOLD_FIRST))


- MAX와 JOIN으로 계산해도 현재 데이터의 결과는 같을 수 있다.
- FIRST_VALUE를 사용한 경우와 SQL 구조를 비교한다.


In [ ]:
ask(
    "각 직원의 이름, 부서번호, 급여와 부서 내 최고 급여와 본인 급여의 차이를 보여줘",
    hint="FIRST_VALUE(sal) OVER (PARTITION BY deptno ORDER BY sal DESC)를 사용한다.",
    gold=GOLD_FIRST,
)


In [ ]:
# 4. OVER ()
GOLD_RATIO = """
SELECT ename, sal,
       ROUND(100.0 * sal / SUM(sal) OVER (), 1) AS pct
FROM emp
"""

ask(
    "각 직원의 이름, 급여와, 전체 급여 합계에서 그 급여가 차지하는 비율(%)을 소수 첫째 자리까지 보여줘",
    gold=GOLD_RATIO,
)

### E. 업무 용어

‘고액 연봉자’를 **급여가 3000 이상인 직원**으로 정의한다. 스키마에 없는 이 기준을 힌트로 제공했을 때와 제공하지 않았을 때의 결과를 비교한다.


In [ ]:
GOLD_HIGH = """
SELECT ename, sal
FROM emp
WHERE sal >= 3000
"""

ask("고액 연봉자의 이름과 급여를 보여줘", gold=GOLD_HIGH)


In [ ]:
ask(
    "고액 연봉자의 이름과 급여를 보여줘",
    hint="'고액 연봉자'는 급여(EMP.sal)가 3000 이상인 직원이다.",
    gold=GOLD_HIGH,
)

업무 용어의 기준이 없으면 LLM이 임의로 해석할 수 있다. 스키마에 없는 정의와 규칙은 힌트로 전달한다.

### F. 재귀 CTE


In [ ]:
GOLD_ORG = """
WITH RECURSIVE org(empno, ename, lvl) AS (
    SELECT empno, ename, 1
    FROM emp
    WHERE ename = 'KING'
    UNION ALL
    SELECT e.empno, e.ename, o.lvl + 1
    FROM emp e JOIN org o ON e.mgr = o.empno
)
SELECT ename, lvl
FROM org
"""

ask(
    "KING부터 시작해서 모든 부하 직원(부하의 부하 포함)의 이름과 단계를 보여줘. "
    "KING은 1단계, KING의 직속 부하는 2단계",
    gold=GOLD_ORG,
)

### G. 질문 작성

- 상사보다 급여가 높은 직원의 이름과 급여
- KING의 두 단계 아래 부하 직원 이름
- 30번 부서의 모든 직원보다 급여가 높은 직원의 이름
- 직원이 없는 부서의 이름
- 직원이 2명 이상인 직무별 평균 급여
- 부서별로 입사일이 가장 빠른 직원
- 부서 내에서 입사일 순서로 계산한 급여 누적 합


In [ ]:
my_question = "각 직원의 이름과 상사의 이름을 보여줘. 상사가 없는 직원도 포함"

ask(my_question)